# Metriplectic Neural Network Training

This notebook implements a Neural Network following **"El Mandato Metriplético"**. 
It combines Symplectic (Hamiltonian) conservative dynamics with Metric (Dissipative) relaxation.

### Core Principles:
1. **Symplectic Component**: $d_{symp} = \{u, H\}$ (Conservative motion using a kinetic energy term and momentum).
2. **Metric Component**: $d_{metr} = [u, S]$ (Dissipative relaxation towards the minimum loss).
3. **Golden Operator ($O_n$)**: Space is modulated by $O_n = \cos(\pi n) \cos(\pi \phi n)$ where $\phi \approx 1.618$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from typing import List, Tuple, Dict

# Constants
PHI = (1 + 5**0.5) / 2

def golden_operator(n):
    return np.cos(np.pi * n) * np.cos(np.pi * PHI * n)

# Activation functions
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return np.where(x > 0, 1, 0)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

class MetriplecticNeuralNetwork:
    def __init__(self, layers: List[int], activation='relu'):
        self.layers = layers
        self.activation_fn = {'relu': relu, 'sigmoid': sigmoid}[activation]
        self.activation_der = {'relu': relu_derivative, 'sigmoid': sigmoid_derivative}[activation]
        
        # Weights and Biases modulated by O_n (Regla 2.1)
        self.weights = []
        self.biases = []
        self.momentum_w = []
        self.momentum_b = []
        
        for i in range(len(layers) - 1):
            n_in, n_out = layers[i], layers[i+1]
            # O_n modulation
            on_factor = golden_operator(i + 1)
            w = np.random.randn(n_in, n_out) * np.sqrt(2./n_in) * on_factor
            b = np.zeros((1, n_out)) + 0.01 * on_factor
            
            self.weights.append(w)
            self.biases.append(b)
            # Momentum for Symplectic part
            self.momentum_w.append(np.zeros_like(w))
            self.momentum_b.append(np.zeros_like(b))

    def compute_lagrangian(self, X, y):
        """Regla 3.1: Returns Symplectic and Metric Lagrangian components."""
        activations, zs = self.forward(X)
        output = activations[-1]
        
        # Potential Energy (Loss)
        loss = 0.5 * np.mean((output - y)**2)
        
        # Kinetic Energy (Sum of squared momentum)
        kinetic = 0.5 * sum(np.sum(p**2) for p in self.momentum_w)
        
        L_symp = kinetic - loss
        L_metr = -loss # Metric part drives towards low S
        return L_symp, L_metr

    def forward(self, X):
        activations = [X]
        zs = []
        curr_a = X
        for w, b in zip(self.weights, self.biases):
            z = np.dot(curr_a, w) + b
            zs.append(z)
            curr_a = self.activation_fn(z)
            activations.append(curr_a)
        return activations, zs

    def step(self, X, y, lr=0.01, kappa=0.05):
        """
        Metriplectic update:
        - Symplectic: Hamiltonian flow (Momentum update)
        - Metric: Gradient dissipation
        """
        activations, zs = self.forward(X)
        output = activations[-1]
        
        # Backprop for Gradient (Metric part)
        error = output - y
        grad_w = []
        grad_b = []
        
        delta = error * self.activation_der(zs[-1])
        for i in reversed(range(len(self.weights))):
            gw = np.dot(activations[i].T, delta)
            gb = np.sum(delta, axis=0, keepdims=True)
            grad_w.insert(0, gw)
            grad_b.insert(0, gb)
            
            if i > 0:
                delta = np.dot(delta, self.weights[i].T) * self.activation_der(zs[i-1])
        
        # Update Rules
        for i in range(len(self.weights)):
            # 1. Symplectic part: dw = p, dp = -dV
            # We use a leapfrog-style or momentum update for conservation
            self.momentum_w[i] = (1 - kappa) * self.momentum_w[i] - lr * grad_w[i]
            self.momentum_b[i] = (1 - kappa) * self.momentum_b[i] - lr * grad_b[i]
            
            self.weights[i] += self.momentum_w[i]
            self.biases[i] += self.momentum_b[i]
            
        return 0.5 * np.mean((output - y)**2), grad_w

def train_and_visualize(model, X, y, epochs=100):
    losses = []
    kinetic_energies = []
    potential_energies = []
    
    for e in range(epochs):
        loss, grads = model.step(X, y)
        losses.append(loss)
        
        l_symp, l_metr = model.compute_lagrangian(X, y)
        potential_energies.append(loss)
        kinetic_energies.append(sum(np.sum(p**2) for p in model.momentum_w))
        
        if e % 20 == 0:
            print(f"Epoch {e}: Loss={loss:.6f}")

    plt.figure(figsize=(10, 5))
    plt.plot(potential_energies, label='Potential Energy (Loss)')
    plt.plot(kinetic_energies, label='Kinetic Energy (Momentum)')
    plt.title('Regla 3.3: Competition between Conservative and Dissipative Terms')
    plt.xlabel('Epoch')
    plt.ylabel('Energy')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# XOR Test Case
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([[0], [1], [1], [0]])

model = MetriplecticNeuralNetwork([2, 8, 1], activation='relu')
train_and_visualize(model, X, y, epochs=200)